In [1]:
from pathlib import Path
import shutil
import yaml


# ============================================================
# 1. НАСТРОЙКИ
# ============================================================

# Исходный датасет
DATASET_PATH = Path(
    r"C:\Users\Ева\Desktop\Хакатон\data_preproc\datasets\large-machinery-v3.v1i.yolo26"
)

# Исходные классы, которые оставляем:
# старый 0 -> новый 0 = excavator
# старый 2 -> новый 1 = crane

CLASS_MAPPING = {
    0: 0,
    2: 1,
}

NEW_CLASS_NAMES = [
    "excavator",
    "crane",
]


# ============================================================
# 2. ПАПКИ
# ============================================================

IMAGES_DIR = DATASET_PATH / "images"
LABELS_DIR = DATASET_PATH / "labels"

if not IMAGES_DIR.exists():
    raise FileNotFoundError(
        f"Не найдена папка images:\n{IMAGES_DIR}"
    )

if not LABELS_DIR.exists():
    raise FileNotFoundError(
        f"Не найдена папка labels:\n{LABELS_DIR}"
    )


# ============================================================
# 3. СОЗДАЁМ BACKUP
# ============================================================

BACKUP_DIR = DATASET_PATH / "backup_before_filtering"

if BACKUP_DIR.exists():

    print(
        f"Backup уже существует:\n{BACKUP_DIR}"
    )

    answer = input(
        "Продолжить без создания нового backup? [Y/N]: "
    )

    if answer.lower() != "y":
        raise RuntimeError(
            "Операция остановлена."
        )

else:

    print("Создаём backup...")

    BACKUP_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copytree(
        IMAGES_DIR,
        BACKUP_DIR / "images"
    )

    shutil.copytree(
        LABELS_DIR,
        BACKUP_DIR / "labels"
    )

    print(
        f"Backup создан:\n{BACKUP_DIR}"
    )


# ============================================================
# 4. НАСТРОЙКИ ФАЙЛОВ
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff",
}


image_files = sorted(
    [
        p
        for p in IMAGES_DIR.rglob("*")
        if (
            p.is_file()
            and
            p.suffix.lower()
            in IMAGE_EXTENSIONS
        )
    ]
)


print()
print("=" * 70)
print("FILTER DATASET")
print("=" * 70)

print(
    f"Dataset: {DATASET_PATH}"
)

print(
    f"Images:  {len(image_files)}"
)

print(
    f"Keeping classes:"
)

print(
    "  0 -> 0: excavator"
)

print(
    "  2 -> 1: crane"
)

print(
    "Removing classes:"
)

print(
    "  1, 3, 4"
)

print()


# ============================================================
# 5. СТАТИСТИКА
# ============================================================

processed_images = 0
images_with_remaining_labels = 0
deleted_images = 0

removed_objects = 0
kept_objects = 0

deleted_label_files = 0

class_stats_before = {}
class_stats_after = {
    0: 0,
    1: 0,
}


# ============================================================
# 6. ОБРАБОТКА LABELS
# ============================================================

for image_path in image_files:

    relative_path = image_path.relative_to(
        IMAGES_DIR
    )

    label_path = (
        LABELS_DIR /
        relative_path.with_suffix(".txt")
    )

    processed_images += 1

    # --------------------------------------------------------
    # Если label отсутствует
    # --------------------------------------------------------

    if not label_path.exists():

        print(
            f"⚠ Нет label: {relative_path}"
        )

        # Изображение без label также удаляем,
        # поскольку после фильтрации оно не имеет разметки.

        image_path.unlink()

        deleted_images += 1

        continue


    # --------------------------------------------------------
    # Читаем label
    # --------------------------------------------------------

    with open(
        label_path,
        "r",
        encoding="utf-8"
    ) as f:

        lines = [
            line.strip()
            for line in f
            if line.strip()
        ]


    new_lines = []


    for line in lines:

        parts = line.split()


        # Некорректная строка
        if len(parts) != 5:

            print(
                f"⚠ Некорректная строка:"
                f"\n  {label_path}"
                f"\n  {line}"
            )

            continue


        try:

            old_class_id = int(parts[0])

        except ValueError:

            print(
                f"⚠ Некорректный class_id:"
                f"\n  {label_path}"
                f"\n  {line}"
            )

            continue


        # ----------------------------------------------------
        # Сохраняем статистику старых классов
        # ----------------------------------------------------

        class_stats_before[
            old_class_id
        ] = (
            class_stats_before.get(
                old_class_id,
                0
            )
            + 1
        )


        # ----------------------------------------------------
        # УДАЛЯЕМ НЕНУЖНЫЕ КЛАССЫ
        # ----------------------------------------------------

        if old_class_id not in CLASS_MAPPING:

            removed_objects += 1

            continue


        # ----------------------------------------------------
        # ПЕРЕНОМЕРУЕМ КЛАСС
        # ----------------------------------------------------

        new_class_id = CLASS_MAPPING[
            old_class_id
        ]

        parts[0] = str(
            new_class_id
        )


        new_line = " ".join(parts)

        new_lines.append(
            new_line
        )


        kept_objects += 1

        class_stats_after[
            new_class_id
        ] += 1


    # ========================================================
    # ЕСЛИ ПОСЛЕ ФИЛЬТРАЦИИ НИЧЕГО НЕ ОСТАЛОСЬ
    # ========================================================

    if len(new_lines) == 0:

        # Удаляем label
        label_path.unlink()

        deleted_label_files += 1

        # Удаляем изображение
        image_path.unlink()

        deleted_images += 1

        continue


    # ========================================================
    # СОХРАНЯЕМ НОВУЮ РАЗМЕТКУ
    # ========================================================

    with open(
        label_path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            "\n".join(new_lines)
        )

    images_with_remaining_labels += 1


# ============================================================
# 7. УДАЛЯЕМ LABELS, ДЛЯ КОТОРЫХ НЕТ IMAGE
# ============================================================

orphan_labels = 0

for label_path in LABELS_DIR.rglob("*.txt"):

    relative_path = label_path.relative_to(
        LABELS_DIR
    )

    image_exists = False

    for extension in IMAGE_EXTENSIONS:

        candidate = (
            IMAGES_DIR /
            relative_path.with_suffix(extension)
        )

        if candidate.exists():

            image_exists = True
            break


    if not image_exists:

        print(
            f"Удаляем label без изображения:"
            f"\n  {label_path}"
        )

        label_path.unlink()

        orphan_labels += 1


# ============================================================
# 8. СОЗДАЁМ НОВЫЙ data.yaml
# ============================================================

DATA_YAML = DATASET_PATH / "data.yaml"

yaml_data = {

    "train": "images",

    "val": "images",

    "test": "images",

    "nc": 2,

    "names": NEW_CLASS_NAMES,

}


with open(
    DATA_YAML,
    "w",
    encoding="utf-8"
) as f:

    yaml.dump(
        yaml_data,
        f,
        allow_unicode=True,
        sort_keys=False
    )


# ============================================================
# 9. ИТОГ
# ============================================================

remaining_images = [
    p
    for p in IMAGES_DIR.rglob("*")
    if (
        p.is_file()
        and
        p.suffix.lower()
        in IMAGE_EXTENSIONS
    )
]

remaining_labels = [
    p
    for p in LABELS_DIR.rglob("*.txt")
    if p.is_file()
]


print()
print("=" * 70)
print("ГОТОВО")
print("=" * 70)

print()

print(
    f"Изображений обработано: "
    f"{processed_images}"
)

print(
    f"Изображений удалено: "
    f"{deleted_images}"
)

print(
    f"Осталось изображений: "
    f"{len(remaining_images)}"
)

print()

print(
    f"Удалено объектов: "
    f"{removed_objects}"
)

print(
    f"Оставлено объектов: "
    f"{kept_objects}"
)

print()

print(
    f"Удалено пустых label: "
    f"{deleted_label_files}"
)

print(
    f"Удалено orphan labels: "
    f"{orphan_labels}"
)

print()

print("КЛАССЫ ДО ФИЛЬТРАЦИИ:")

for class_id in sorted(class_stats_before):

    print(
        f"  {class_id}: "
        f"{class_stats_before[class_id]}"
    )


print()

print("КЛАССЫ ПОСЛЕ ФИЛЬТРАЦИИ:")

print(
    f"  0: excavator — "
    f"{class_stats_after[0]}"
)

print(
    f"  1: crane     — "
    f"{class_stats_after[1]}"
)

print()

print("Новый data.yaml:")

print(
    DATA_YAML.resolve()
)

print()

print("Backup:")

print(
    BACKUP_DIR.resolve()
)

Создаём backup...
Backup создан:
C:\Users\Ева\Desktop\Хакатон\data_preproc\datasets\large-machinery-v3.v1i.yolo26\backup_before_filtering

FILTER DATASET
Dataset: C:\Users\Ева\Desktop\Хакатон\data_preproc\datasets\large-machinery-v3.v1i.yolo26
Images:  2495
Keeping classes:
  0 -> 0: excavator
  2 -> 1: crane
Removing classes:
  1, 3, 4


ГОТОВО

Изображений обработано: 2495
Изображений удалено: 25
Осталось изображений: 2470

Удалено объектов: 1624
Оставлено объектов: 4219

Удалено пустых label: 25
Удалено orphan labels: 0

КЛАССЫ ДО ФИЛЬТРАЦИИ:
  0: 1360
  1: 15
  2: 2859
  3: 543
  4: 1066

КЛАССЫ ПОСЛЕ ФИЛЬТРАЦИИ:
  0: excavator — 1360
  1: crane     — 2859

Новый data.yaml:
C:\Users\Ева\Desktop\Хакатон\data_preproc\datasets\large-machinery-v3.v1i.yolo26\data.yaml

Backup:
C:\Users\Ева\Desktop\Хакатон\data_preproc\datasets\large-machinery-v3.v1i.yolo26\backup_before_filtering
